<a href="https://colab.research.google.com/github/danielbr9382/Corpus2GeoRAG_Pacific/blob/main/Processing/colab_ner.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# NER + DATE/TIME for the GeoRAG corpus
Upload `DATA.json` (or a JSON/JSONL file containing text records). The notebook runs the known BERT baseline for PER/ORG/LOC/MISC and spaCy for DATE/TIME. DATE/TIME are text mentions, not validated event dates.

Start with `LIMIT = 2`, inspect the outputs, then set `LIMIT = None` for the full corpus.


In [61]:
%pip -q install transformers spacy torch
!python -m spacy download en_core_web_sm


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 55.0 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [62]:
import json
import subprocess
from pathlib import Path
import transformers
import spacy
import torch

הגדרת תיקיות

In [63]:
repo_dir = Path("/content/Corpus2GeoRAG_Pacific")

if not repo_dir.exists():
    subprocess.run(
        ["git", "clone", "--branch", "main",
         "https://github.com/danielbr9382/Corpus2GeoRAG_Pacific.git",
         str(repo_dir)],
        check=True,
    )

In [64]:
data_dir = repo_dir / "Data"
json_files = list(data_dir.glob("*.json"))
if len(json_files) != 1:
    raise ValueError(f"ציפיתי לקובץ JSON אחד בתיקיית Data; נמצאו: {[p.name for p in json_files]}")

קריאה לDATA

In [65]:
input_path = json_files[0]
output_dir = data_dir / "TMP_NER"

with input_path.open(encoding="utf-8") as f:
    data = json.load(f)

print(f"נטען: {input_path.name} — {len(data)} מסמכים")
print(f"פלט יישמר ב: {output_dir}")

נטען: DATA.json — 428 מסמכים
פלט יישמר ב: /content/Corpus2GeoRAG_Pacific/Data/TMP_NER


הצגת 3 הKEYS הראשונים

In [66]:
print(list(data.keys())[:3]) # שלוש הכותרות הראשונות

['14th Indian Infantry Division', '1939–1940 Winter Offensive', '1943 Cairo Declaration']


הגרלת 100 פסקאות

In [67]:
import json
import random
import re

SEED = 42
SAMPLE_SIZE = 100

paragraphs = []

for title, text in data.items():
    separators = list(re.finditer(r"\n[ \t]*\n+", text))
    starts = [0] + [m.end() for m in separators]
    ends = [m.start() for m in separators] + [len(text)]

    for index, (left, right) in enumerate(zip(starts, ends)):
        chunk = text[left:right]
        paragraph = chunk.strip()

        if not paragraph or re.fullmatch(r"=+\s*.*?\s*=+", paragraph):
            continue

        start = left + len(chunk) - len(chunk.lstrip())
        end = left + len(chunk.rstrip())

        paragraphs.append({
            "article_title": title,
            "paragraph_index": index,
            "start": start,
            "end": end,
            "text": text[start:end],
        })

paragraphs.sort(key=lambda p: (p["article_title"], p["start"]))

if SAMPLE_SIZE > len(paragraphs):
    raise ValueError(f"ביקשת {SAMPLE_SIZE} פסקאות, אך נמצאו רק {len(paragraphs)}")

sample = random.Random(SEED).sample(paragraphs, SAMPLE_SIZE)

output_dir.mkdir(parents=True, exist_ok=True)
sample_path = output_dir / f"sample_{SAMPLE_SIZE}_paragraphs_seed{SEED}.json"

result = {
    "source_file": input_path.name,
    "seed": SEED,
    "sample_size": SAMPLE_SIZE,
    "paragraphs": sample,
}

with sample_path.open("w", encoding="utf-8") as f:
    json.dump(result, f, ensure_ascii=False, indent=2)

print(f"נשמרו {len(sample)} פסקאות מתוך {len(paragraphs)}")
print(sample_path)

נשמרו 100 פסקאות מתוך 9496
/content/Corpus2GeoRAG_Pacific/Data/TMP_NER/sample_100_paragraphs_seed42.json


העלאה לGIT

In [68]:
# import base64
# import requests
# from google.colab import userdata

# token = userdata.get("COLAB")
# if not token:
#     raise ValueError("לא נמצא GITHUB_TOKEN ב-Colab Secrets")

# repo_path = f"Data/TMP_NER/{sample_path.name}"
# url = f"https://api.github.com/repos/danielbr9382/Corpus2GeoRAG_Pacific/contents/{repo_path}"
# headers = {
#     "Authorization": f"Bearer {token}",
#     "Accept": "application/vnd.github+json",
# }
# content = sample_path.read_bytes()

# existing = requests.get(url, headers=headers, params={"ref": "main"}, timeout=30)
# if existing.status_code == 200:
#     remote_content = base64.b64decode(existing.json()["content"])
#     if remote_content != content:
#         raise FileExistsError("ב-GitHub כבר קיים קובץ בשם הזה עם תוכן שונה")
#     print("הקובץ הזה כבר נמצא ב-GitHub")
# elif existing.status_code == 404:
#     response = requests.put(
#         url,
#         headers=headers,
#         json={
#             "message": f"Add NER sample {sample_path.name}",
#             "content": base64.b64encode(content).decode("ascii"),
#             "branch": "main",
#         },
#         timeout=60,
#     )
#     response.raise_for_status()
#     print("הקובץ הועלה ל-GitHub")
# else:
#     existing.raise_for_status()

# print(f"https://github.com/danielbr9382/Corpus2GeoRAG_Pacific/blob/main/{repo_path}")

הקובץ הזה כבר נמצא ב-GitHub
https://github.com/danielbr9382/Corpus2GeoRAG_Pacific/blob/main/Data/TMP_NER/sample_100_paragraphs_seed42.json


In [69]:
# error = response.json()

# print("status:", response.status_code)
# print("message:", error.get("message"))
# print("required permissions:",
#       response.headers.get("X-Accepted-GitHub-Permissions"))
# print("rate limit remaining:",
#       response.headers.get("X-RateLimit-Remaining"))

status: 201
message: None
required permissions: contents=write; contents=write,workflows=write
rate limit remaining: 4993


In [ ]:
from pathlib import Path
from shutil import copy2

sample_path = Path(sample_path)
output_dir = Path("/content/TMP_NER")
output_dir.mkdir(parents=True, exist_ok=True)

saved_path = output_dir / sample_path.name
if sample_path.resolve() != saved_path.resolve():
    copy2(sample_path, saved_path)

print(f"המדגם נשמר ב־Colab: {saved_path}")

בחינת מודלי NER